# Compute GPT-2 Perplexity for Unified Review Dataset

This Google Colab notebook computes token-level perplexity scores using **GPT-2** for all review records in `unified_dataset.csv`.
Scores are keyed by `row_id` and exported to `perplexity.csv` for seamless integration into the stylometric detection pipeline.

In [ ]:
# Check GPU availability
!nvidia-smi
!pip install --quiet transformers torch pandas tqdm

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from tqdm.auto import tqdm
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

model_id = 'gpt2'
tokenizer = GPT2TokenizerFast.from_pretrained(model_id)
model = GPT2LMHeadModel.from_pretrained(model_id).to(device)
model.eval()

In [ ]:
# Upload or load unified_dataset.csv
csv_path = 'unified_dataset.csv'
try:
    df = pd.read_csv(csv_path)
    print(f'Loaded {len(df)} reviews with columns: {list(df.columns)}')
except FileNotFoundError:
    print(f'Please upload {csv_path} to Colab working directory.')

In [ ]:
def compute_perplexity(text, max_length=512):
    if not isinstance(text, str) or not text.strip():
        return 1e6
    encodings = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_length)
    input_ids = encodings.input_ids.to(device)
    target_ids = input_ids.clone()
    if input_ids.shape[1] < 2:
        return 1e6
    with torch.no_grad():
        outputs = model(input_ids, labels=target_ids)
        neg_log_likelihood = outputs.loss.item()
    ppl = math.exp(neg_log_likelihood) if neg_log_likelihood < 100 else 1e6
    return float(ppl)

perplexities = []
for text in tqdm(df['text'], desc='Calculating GPT-2 Perplexity'):
    perplexities.append(compute_perplexity(text))

out_df = pd.DataFrame({
    'row_id': df['row_id'],
    'perplexity': perplexities
})
out_path = 'perplexity.csv'
out_df.to_csv(out_path, index=False)
print(f'Saved {len(out_df)} perplexity scores to {out_path}')

In [ ]:
# Download perplexity.csv
from google.colab import files
files.download('perplexity.csv')
print('Place downloaded perplexity.csv into data/processed/perplexity.csv to activate it.')